In [2]:
import os
import rasterio
import json
import numpy as np

import matplotlib.pyplot as plt
from scipy.optimize import bisect

In [3]:
def displacement(logky, logpga, M=None, logpgv=None, model="scalar"):
    """ Calculation of ground displacements
    Implementation of the statistical model for ground displacements of natural slopes subject to earthquakes 
    given in [1]. Note: The statistical model is develpped for subaerial conditions.
    
    Parameters:
        logky: float or ndarray 
            Log (Base 10) of Yield acceleration calculated using infinite slope analysis [g].
        logpga: float or ndarray. Same dimension as ky.
            Log (Base 10) of Peak ground acceleration of the event [g]. 
        M: float
            moment magnitude of the event.
        logpgv: float or ndarray. Same dimension as ky.
            Log (Base 10) of Peak ground velocity of the event [cm/s].
        model: string
            Different models. Options are "scalar" or "vector". If "scalar", then M must be supplied. 
            If "vector", then pgv must be supplied. Default is "scalar".
        
    Returns:
        Log of displacements [cm], Log of standard_deviation: float or ndarray, float or ndarray
        Logarithm of estimated displacements and associated standard deviation. 
        Standard deviation of lognormal multiplicative noise (as a function of ky/pga).  
        
    1. Rathje and Saygili, ‘Probabilistic Assessment of Earthquake-Induced Sliding Displacements of Natural Slopes’.
    """
    ky_pga_ratio = 10**(logky - logpga)
    logscale_factor = np.log10(np.exp(1))
    lnpga = logpga/logscale_factor
    
    if model == "scalar":
        a = [-29.06, 42.49, - 19.64,-4.85, 4.89] # polynomial coefficients.
        ln_d = np.where(ky_pga_ratio < 1., np.polyval(a, ky_pga_ratio) + 0.72*lnpga + 0.89*(M-6), 0.)
        sigma_ln = np.where(ky_pga_ratio < 1., np.polyval([-0.539, 0.789, 0.732], ky_pga_ratio), 0.)
    elif model == "vector":
        lnpgv = logpgv/np.log10(np.exp(1))
        a = [-30.5, 44.75, -20.84, -4.58, -1.56]
        ln_d = np.where(ky_pga_ratio < 1., np.polyval(a, ky_pga_ratio) - 0.64*lnpga + 1.55*lnpgv, 0)
        sigma_ln = np.where(ky_pga_ratio, 0.405 + 0.524*(ky_pga_ratio), 0.)
    return(ln_d*logscale_factor, sigma_ln*logscale_factor)

In [4]:
def f(logky):
    logd, logsigma = displacement(logky=logky, logpga=logpga, M=M)
    return(logd - np.log10(delta))

In [ ]:
logpga = -0.6
M = 8.

selection_criteria = []
for delta in [3., 5., 10., 50]:
    root_logky = bisect(f, -2, 1)
    print(root_logky)
    
    selection_criteria.append({"root_logky": root_logky, "delta": delta, "logpga": logpga, "M":M})

In [ ]:
logpgas = np.log10([0.6, 1, 1.5])
delta = 5.
print("pga: {}".format(10**logpga))
M = 8.
logky = np.linspace(-5,1,200000)

selection_criteria = []
for logpga in logpgas:
    #print(displacement(logky=logky, logpga=logpga, M=M)[0])
    more_than_delta = displacement(logky=logky, logpga=logpga, M=M)[0] > np.log10(delta)
    root_logky = logky[more_than_delta][-1] # Biggest log yield acelleration with displacement larger than delta.
    
    selection_criteria.append({"root_logky": root_logky, "delta": delta, "logpga": logpga, "M":M})

In [ ]:
selection_criteria

In [ ]:
# Extract areas using the Cummulative distribution of logky.
np.array([params["root_logky"] + params["logpga"] for params in selection_criteria])

## Intersect slopeunits to create nested sets of volumes.

In [10]:
def read_tif(fname):
    "Read .tif data and profile using rasterio."
    #logger.info(f"Read file: {fname}")
    with rasterio.open(fname) as src:
        #data = np.ma.masked_equal(src.read(1), src.nodata)
        data = src.read(1)
        msk = np.where(src.read_masks(1) == src.nodata, False, True)
        profile = src.profile.copy()
    return data, msk, profile

In [111]:
# load json file
cumulative_dir = "/home/ebr/projects/release-volume-sampler/generated/messina_001/volume_selection/cummulative"
volumes_dir = "/home/ebr/projects/release-volume-sampler/generated/messina_001/volume_selection"

with open(os.path.join(volumes_dir, 'selection_criteria.json'), 'r') as f:
    selection_criteria = json.load(f)

with open(os.path.join(cumulative_dir, "content.json"),'r') as f:
    content = json.load(f)

rasters = []
# Read all rasters and store the data
for i,c in enumerate(content):
    raster_path = os.path.join(cumulative_dir, c["file"])
    raster_data, msk, profile = read_tif(raster_path)
    rasters.append(raster_data)


In [ ]:
bathy = "/home/ebr/projects/release-volume-sampler/input/bathy/messina_001/bathy_truncated.tif"
slopeunits_file = "/home/ebr/projects/release-volume-sampler/generated/slopeunits/slumap_clean_utm.tif"
with rasterio.open() as src:
    print(src.crs, src.bounds)

In [ ]:
selection_criteria

In [66]:

slopeunits, msk, profile = read_tif("/home/ebr/projects/release-volume-sampler/generated/slopeunits/slumap_clean.tif")

In [ ]:
plt.imshow(rasters[2] > 0.3)

In [ ]:
plt.imshow(slopeunits, vmin=0, vmax=1000)

In [ ]:
np.where(slopeunits == 1, True, False)

In [ ]:
profile